# MODEL COMPARISON

Each model uses the same chronological train, validation, and test partitions. Tunable models are selected only from validation MASE, then refitted on train + validation, and the test set is evaluated once. Linear Regression is reported as a fixed ordinary least-squares baseline.

In [1]:
import time
import warnings

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeRegressor

# DATA

In [ ]:
data_path = "/tmp/deep_generic_smoke.csv"
target_name = "y"
date_column_name = "date"
test_fraction = 0.30
validation_fraction = 0.30
mase_lag = 1
search_iterations = 10000
random_state = 0

if not data_path:
    raise ValueError("Set data_path to the time-ordered CSV file before running.")

df = pd.read_csv(data_path)
for optional_column in ("Unnamed: 0", date_column_name):
    if optional_column in df.columns:
        df = df.drop(columns=optional_column)

if target_name not in df.columns:
    raise ValueError(f"Target column {target_name!r} was not found.")
if df.isna().any().any():
    raise ValueError("The data contain missing values.")

y = df[target_name].astype(float).reset_index(drop=True)
X = df.drop(columns=target_name).astype(float).reset_index(drop=True)
if not np.isfinite(X.to_numpy()).all() or not np.isfinite(y.to_numpy()).all():
    raise ValueError("The data contain non-finite values.")

test_len = int(np.ceil(len(y) * test_fraction))
train_validation_len = len(y) - test_len
validation_len = int(np.ceil(train_validation_len * validation_fraction))
train_len = train_validation_len - validation_len
if train_len <= mase_lag or validation_len < 1 or test_len < 1:
    raise ValueError("The dataset is too short for the requested split.")

X_train_validation = X.iloc[:train_validation_len]
y_train_validation = y.iloc[:train_validation_len]
X_test = X.iloc[train_validation_len:]
y_test = y.iloc[train_validation_len:]

print(X.shape)
print(f"Train {train_len}")
print(f"Validation {validation_len}")
print(f"Test {test_len}")

(40, 2)
Train 19
Validation 9
Test 12


# METRICS AND RANDOMIZED SEARCH

In [6]:
def calculate_metrics(y_true, y_pred, y_scaling, lag=1):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_scaling = np.asarray(y_scaling, dtype=float)
    error = y_true - y_pred
    mae_score = float(np.mean(np.abs(error)))
    mse_score = float(np.mean(error ** 2))
    nonzero = np.abs(y_true) > np.finfo(float).eps
    mape_score = float(np.mean(np.abs(error[nonzero] / y_true[nonzero]))) if nonzero.any() else np.nan
    mase_denominator = float(np.mean(np.abs(y_scaling[lag:] - y_scaling[:-lag])))
    target_range = float(np.max(y_scaling) - np.min(y_scaling))
    mase_score = mae_score / mase_denominator if mase_denominator > np.finfo(float).eps else np.nan
    nmae_score = mae_score / target_range if target_range > np.finfo(float).eps else np.nan
    return {
        "MASE": mase_score,
        "nMAE": nmae_score,
        "MSE": mse_score,
        "MAPE": mape_score,
        "MAE": mae_score,
    }


def run_randomized_model(model_name, model, search_space):
    test_fold = np.concatenate([
        np.full(train_len, -1, dtype=int),
        np.zeros(validation_len, dtype=int),
    ])
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=search_space,
        n_iter=search_iterations,
        scoring="neg_mean_absolute_error",
        cv=PredefinedSplit(test_fold),
        random_state=random_state,
        n_jobs=-1,
        refit=False,
        error_score=np.nan,
    )

    start_time = time.perf_counter()
    search.fit(X_train_validation, y_train_validation)
    search_elapsed_time = time.perf_counter() - start_time

    validation_target = y.iloc[:train_len].to_numpy()
    validation_scale = float(np.mean(np.abs(validation_target[mase_lag:] - validation_target[:-mase_lag])))
    validation_mase = -float(search.best_score_) / validation_scale
    final_model = clone(model).set_params(**search.best_params_)

    start_time = time.perf_counter()
    final_model.fit(X_train_validation, y_train_validation)
    training_elapsed_time = time.perf_counter() - start_time

    start_time = time.perf_counter()
    predictions = final_model.predict(X_test)
    test_elapsed_time = time.perf_counter() - start_time

    results = {
        "Model": model_name,
        "Best Parameters": search.best_params_,
        "Search Iterations": search_iterations,
        "Validation MASE": validation_mase,
        **calculate_metrics(y_test, predictions, y_train_validation, mase_lag),
        "Search Time": search_elapsed_time,
        "Training Time": training_elapsed_time,
        "Test Time": test_elapsed_time,
        "One Prediction Time": test_elapsed_time / test_len,
        "Total Time": training_elapsed_time + test_elapsed_time,
    }
    return results, final_model, np.asarray(predictions)


def run_fixed_model(model_name, model):
    validation_model = clone(model)
    validation_model.fit(X.iloc[:train_len], y.iloc[:train_len])
    validation_predictions = validation_model.predict(X.iloc[train_len:train_validation_len])
    validation_mase = calculate_metrics(
        y.iloc[train_len:train_validation_len],
        validation_predictions,
        y.iloc[:train_len],
        mase_lag,
    )["MASE"]

    final_model = clone(model)
    start_time = time.perf_counter()
    final_model.fit(X_train_validation, y_train_validation)
    training_elapsed_time = time.perf_counter() - start_time

    start_time = time.perf_counter()
    predictions = final_model.predict(X_test)
    test_elapsed_time = time.perf_counter() - start_time

    results = {
        "Model": model_name,
        "Best Parameters": {},
        "Search Iterations": 0,
        "Validation MASE": validation_mase,
        **calculate_metrics(y_test, predictions, y_train_validation, mase_lag),
        "Search Time": 0.0,
        "Training Time": training_elapsed_time,
        "Test Time": test_elapsed_time,
        "One Prediction Time": test_elapsed_time / test_len,
        "Total Time": training_elapsed_time + test_elapsed_time,
    }
    return results, final_model, np.asarray(predictions)


all_results = []

# LIGHTGBM

In [7]:
lgbm_model = Pipeline([
    ("scale", MinMaxScaler()),
    ("model", lgb.LGBMRegressor(random_state=random_state, verbose=-1, n_jobs=1)),
])
lgbm_search_space = {
    "model__n_estimators": [100, 250, 500, 750, 1000],
    "model__learning_rate": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1],
    "model__num_leaves": [15, 31, 63, 127, 255],
    "model__max_depth": [4, 6, 7, 8, 10],
    "model__subsample": [0.4, 0.6, 0.7, 0.9],
    "model__subsample_freq": [1, 5, 10, 20, 50],
    "model__colsample_bytree": [0.4, 0.6, 0.7, 0.9],
    "model__reg_alpha": [0, 0.01, 0.05, 0.5, 1, 10],
    "model__reg_lambda": [0, 0.01, 0.05, 0.5, 1, 10],
    "model__max_bin": [15, 31, 63, 127, 255],
}
results, lgbm_model, lgbm_preds = run_randomized_model("LightGBM", lgbm_model, lgbm_search_space)
all_results.append(results)
print(results)

{'Model': 'LightGBM', 'Best Parameters': {'model__subsample_freq': 50, 'model__subsample': 0.4, 'model__reg_lambda': 1, 'model__reg_alpha': 0.5, 'model__num_leaves': 127, 'model__n_estimators': 500, 'model__max_depth': 7, 'model__max_bin': 63, 'model__learning_rate': 0.01, 'model__colsample_bytree': 0.7}, 'Search Iterations': 1, 'Validation MASE': 3.0566416314371256, 'MASE': 7.011227872929274, 'nMAE': 0.7058441528639235, 'MSE': 0.6360079881066164, 'MAPE': 0.5672247765014484, 'MAE': 0.7764285681503159, 'Search Time': 0.8828926379937911, 'Training Time': 0.02731298399157822, 'Test Time': 0.0022495060111396015, 'One Prediction Time': 0.00018745883426163346, 'Total Time': 0.029562490002717823}


# XGBOOST

In [8]:
xgb_model = Pipeline([
    ("scale", MinMaxScaler()),
    ("model", xgb.XGBRegressor(random_state=random_state, tree_method="hist", verbosity=0, n_jobs=1)),
])
xgb_search_space = {
    "model__n_estimators": [100, 250, 500, 750, 1000],
    "model__learning_rate": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1],
    "model__max_depth": [3, 4, 6, 7, 8, 10],
    "model__subsample": [0.4, 0.6, 0.7, 0.9, 1.0],
    "model__colsample_bytree": [0.4, 0.6, 0.7, 0.9, 1.0],
    "model__min_child_weight": [1, 3, 5, 10],
    "model__gamma": [0, 0.01, 0.1, 0.5, 1],
    "model__reg_alpha": [0, 0.01, 0.05, 0.5, 1, 10],
    "model__reg_lambda": [0, 0.01, 0.05, 0.5, 1, 10],
    "model__max_bin": [31, 63, 127, 255],
}
results, xgb_model, xgb_preds = run_randomized_model("XGBoost", xgb_model, xgb_search_space)
all_results.append(results)
print(results)

{'Model': 'XGBoost', 'Best Parameters': {'model__subsample': 1.0, 'model__reg_lambda': 1, 'model__reg_alpha': 0.05, 'model__n_estimators': 1000, 'model__min_child_weight': 5, 'model__max_depth': 6, 'model__max_bin': 63, 'model__learning_rate': 0.05, 'model__gamma': 0.5, 'model__colsample_bytree': 0.7}, 'Search Iterations': 1, 'Validation MASE': 2.6779561633155464, 'MASE': 5.872723693433016, 'nMAE': 0.5912270654331554, 'MSE': 0.4561214925765145, 'MAPE': 0.47209490049523967, 'MAE': 0.650349771976471, 'Search Time': 0.8617114929948002, 'Training Time': 0.11944718100130558, 'Test Time': 0.002461316005792469, 'One Prediction Time': 0.0002051096671493724, 'Total Time': 0.12190849700709805}


# DECISION TREE

In [9]:
decision_tree_model = Pipeline([
    ("scale", MinMaxScaler()),
    ("model", DecisionTreeRegressor(random_state=random_state)),
])
decision_tree_search_space = {
    "model__criterion": ["squared_error", "friedman_mse", "absolute_error"],
    "model__splitter": ["best", "random"],
    "model__max_depth": [2, 3, 4, 5, 6, 7, 8, 10, 12, 15, None],
    "model__min_samples_split": [2, 3, 4, 5, 8, 10, 15, 20, 30, 50],
    "model__min_samples_leaf": [1, 2, 3, 4, 5, 8, 10],
    "model__max_features": [None, "sqrt", "log2", 0.5, 0.8],
    "model__max_leaf_nodes": [None, 8, 16, 32, 64, 128],
}
results, decision_tree_model, decision_tree_preds = run_randomized_model("Decision Tree", decision_tree_model, decision_tree_search_space)
all_results.append(results)
print(results)

{'Model': 'Decision Tree', 'Best Parameters': {'model__splitter': 'random', 'model__min_samples_split': 5, 'model__min_samples_leaf': 2, 'model__max_leaf_nodes': 128, 'model__max_features': 'sqrt', 'model__max_depth': None, 'model__criterion': 'squared_error'}, 'Search Iterations': 1, 'Validation MASE': 3.4333333333333336, 'MASE': 4.901337792642139, 'nMAE': 0.49343434343434334, 'MSE': 0.3240648148148148, 'MAPE': 0.39216662989805545, 'MAE': 0.5427777777777777, 'Search Time': 0.718932167001185, 'Training Time': 0.0020142650027992204, 'Test Time': 0.000663270999211818, 'One Prediction Time': 5.52725832676515e-05, 'Total Time': 0.0026775360020110384}


# LINEAR REGRESSION

In [10]:
linear_model = Pipeline([
    ("scale", MinMaxScaler()),
    ("model", LinearRegression()),
])
results, linear_model, linear_preds = run_fixed_model("Linear Regression", linear_model)
all_results.append(results)
print(results)

{'Model': 'Linear Regression', 'Best Parameters': {}, 'Search Iterations': 0, 'Validation MASE': 23.56722649871567, 'MASE': 2.893604168589315, 'nMAE': 0.2913089718546145, 'MSE': 0.14952676230424966, 'MAPE': 0.22054165905734724, 'MAE': 0.320439869040076, 'Search Time': 0.0, 'Training Time': 0.0026341060001868755, 'Test Time': 0.0010036829917225987, 'One Prediction Time': 8.364024931021656e-05, 'Total Time': 0.003637788991909474}


# NAIVE

In [11]:
naive_forecast = y.shift(1)
validation_preds = naive_forecast.iloc[train_len:train_validation_len].to_numpy()
validation_mase = calculate_metrics(
    y.iloc[train_len:train_validation_len],
    validation_preds,
    y.iloc[:train_len],
    mase_lag,
)["MASE"]

start_time = time.perf_counter()
naive_preds = naive_forecast.iloc[train_validation_len:].to_numpy()
test_elapsed_time = time.perf_counter() - start_time
results = {
    "Model": "Naive",
    "Best Parameters": {"lag": 1},
    "Search Iterations": 0,
    "Validation MASE": validation_mase,
    **calculate_metrics(y_test, naive_preds, y_train_validation, mase_lag),
    "Search Time": 0.0,
    "Training Time": 0.0,
    "Test Time": test_elapsed_time,
    "One Prediction Time": test_elapsed_time / test_len,
    "Total Time": test_elapsed_time,
}
all_results.append(results)
print(results)

{'Model': 'Naive', 'Best Parameters': {'lag': 1}, 'Search Iterations': 0, 'Validation MASE': 0.8476190476190479, 'MASE': 0.8804347826086955, 'nMAE': 0.08863636363636362, 'MSE': 0.013025000000000002, 'MAPE': 0.0740445898573311, 'MAE': 0.09749999999999999, 'Search Time': 0.0, 'Training Time': 0.0, 'Test Time': 0.00016092999430838972, 'One Prediction Time': 1.3410832859032476e-05, 'Total Time': 0.00016092999430838972}


# ANN

In [12]:
ann_regressor = Pipeline([
    ("scale", MinMaxScaler()),
    ("model", MLPRegressor(random_state=random_state, shuffle=False)),
])
ann_model = TransformedTargetRegressor(
    regressor=ann_regressor,
    transformer=MinMaxScaler(),
)
ann_search_space = {
    "regressor__model__hidden_layer_sizes": [(16,), (32,), (64,), (128,), (64, 32), (128, 64), (128, 64, 32)],
    "regressor__model__activation": ["relu", "tanh", "logistic"],
    "regressor__model__solver": ["adam", "sgd"],
    "regressor__model__learning_rate": ["constant", "invscaling", "adaptive"],
    "regressor__model__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 1e-1],
    "regressor__model__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],
    "regressor__model__batch_size": [16, 32, 64, 128, "auto"],
    "regressor__model__max_iter": [500, 1000, 2000, 3500],
}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    results, ann_model, ann_preds = run_randomized_model("ANN", ann_model, ann_search_space)
all_results.append(results)
print(results)

{'Model': 'ANN', 'Best Parameters': {'regressor__model__solver': 'adam', 'regressor__model__max_iter': 2000, 'regressor__model__learning_rate_init': 0.0005, 'regressor__model__learning_rate': 'constant', 'regressor__model__hidden_layer_sizes': (128, 64), 'regressor__model__batch_size': 64, 'regressor__model__alpha': 1e-05, 'regressor__model__activation': 'logistic'}, 'Search Iterations': 1, 'Validation MASE': 3.237684040161609, 'MASE': 7.042658414450605, 'nMAE': 0.7090083723638825, 'MSE': 0.6416350497200125, 'MAPE': 0.5697908372756607, 'MAE': 0.7799092096002708, 'Search Time': 0.7097844319941942, 'Training Time': 0.022169492003740743, 'Test Time': 0.0019127739942632616, 'One Prediction Time': 0.0001593978328552718, 'Total Time': 0.024082265998004004}


# ALL RESULTS

In [13]:
results_table = pd.DataFrame(all_results).sort_values("MASE").reset_index(drop=True)
print(results_table[[
    "Model", "MASE", "nMAE", "MSE", "MAPE", "MAE",
    "Search Time", "Training Time", "Test Time",
    "One Prediction Time", "Total Time",
]].to_string(index=False))

            Model     MASE     nMAE      MSE     MAPE      MAE  Search Time  Training Time  Test Time  One Prediction Time  Total Time
            Naive 0.880435 0.088636 0.013025 0.074045 0.097500     0.000000       0.000000   0.000161             0.000013    0.000161
Linear Regression 2.893604 0.291309 0.149527 0.220542 0.320440     0.000000       0.002634   0.001004             0.000084    0.003638
    Decision Tree 4.901338 0.493434 0.324065 0.392167 0.542778     0.718932       0.002014   0.000663             0.000055    0.002678
          XGBoost 5.872724 0.591227 0.456121 0.472095 0.650350     0.861711       0.119447   0.002461             0.000205    0.121908
         LightGBM 7.011228 0.705844 0.636008 0.567225 0.776429     0.882893       0.027313   0.002250             0.000187    0.029562
              ANN 7.042658 0.709008 0.641635 0.569791 0.779909     0.709784       0.022169   0.001913             0.000159    0.024082
